In [61]:
# Initialize Otter
import otter
grader = otter.Notebook("week8.ipynb")

# HW8: Bayes and Bayes Nets
#### CS-31: Foundations for Artificial Intelligence
*Point breakdown: 50 autograded · 40 manually graded · 10 collaboration statement = 100*



*You are allowed to work in groups for the lab. You must write anything that will be turned in -- all code and all written solutions -- on your own. You must list all people and resources that you interacted with in the Collaboration Statement below. See [course policies](https://www.cs.tufts.edu/comp/31/policy.html) for more information.*

--

In this assignment, you'll be building a simple [Naive Bayes Classifier](https://en.wikipedia.org/wiki/Naive_Bayes_classifier) to classify messages as spam or not spam by estimating probabilities from an existing dataset. In the second part of the assignment, you'll be using libraries to implement direct sampling and rejection sampling on Bayes Nets.

### **Table of Contents**

1) Naive Bayes Classifier
2) Bayes Nets

**AI / Source Disclosure**
- All prose was written by me.
- I used DeepSeek to help with the data loading code for Section 1.0, as well as interpreting the pgmpy code for the sprinkler network in section 2.


## **1. A Naive Bayes Classifier**

In lecture, we learned about *Bayes' rule*

$$P(A|B) = \frac{P(B|A)P(A) }{P(B)}$$

A common way that Bayes' rule is used is that it allows us to calculate the probability of a *cause* given an *effect*. Sometimes this is also stated as predicting the probability of a *hypothesis H* given *data D*.

$$P(H|D) = \frac{P(D|H) P(H) }{P(D)}$$

Let's consider the problem of *classification*. A classification problem is a type of problem where we want to assign labels to data. One classification problem that runs in the background of every networked device you use that you probably don't think about is that of *spam detection*.

Given an incoming e-mail or text message, it would be very useful if we could automatically *classify* it as spam or a legitimate message, such that we could filter out unwanted e-mails to avoid clogging up a user's inbox (or falling for scams). We could write a deterministic algorithm that automatically filters e-mails that contain unwanted and unsavory words (e.g. the e-mail contains the names of certain... adult-oriented medications). However, hand-coding all these rules would be difficult and time-consuming. In addition, we may have some words that often show up in spam e-mails, but also show up in legitimate e-mails as well (e.g. "Click here for a **free** prize!" vs. "Are you **free** for lunch tomorrow"?)

What would be helpful would be to model this process *probabilistically* and estimate, given that an e-mail contains a certain combination of words, the likelihood that it is spam or not. It would also be good if we could estimate these probabilities from data, so we don't have to spend so much time hand-coding rules.

In this case, our *hypothesis* would be "This e-mail is of this class $C$ (Spam or Legitimate)" and our *data* would be the message $m$ that we wish to classify. 

We can model this using Bayes' rule.

$$P(C|m) = \frac{P(m|C) P(C) }{P(m)}$$

So: now our problem is, how do we get values for the three terms on the right side of the equation? Let's walk through it together.

### 1.0 Getting Data

To build our classifier and estimate probabilities, we're going to be using some real data pulled from the [UCI SMS Spam Collection dataset](https://archive.ics.uci.edu/dataset/228/sms+spam+collection). Note that this is real data collected from actual text messages. Some of these messages have adult language, so be advised...

In [62]:
data = {"spam": [], "legit": []}

with open("SMSSpamCollection", encoding="utf-8", errors="ignore") as f:
    for line in f:
        label, _, message = line.partition("\t")
        label = label.strip().lower()
        message = message.strip().lower()
        if label == "spam":
            data["spam"].append(message.split())
        elif label == "ham":
            data["legit"].append(message.split())

We're pre-splitting the data into lists of strings for you. In addition, we're making sure all strings are lowercase to standardize the data even more.

In [63]:
# Viewing some of the legit data...
print(data['legit'][12])
print(data['legit'])

['is', 'that', 'seriously', 'how', 'you', 'spell', 'his', 'name?']
[['go', 'until', 'jurong', 'point,', 'crazy..', 'available', 'only', 'in', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet...', 'cine', 'there', 'got', 'amore', 'wat...'], ['ok', 'lar...', 'joking', 'wif', 'u', 'oni...'], ['u', 'dun', 'say', 'so', 'early', 'hor...', 'u', 'c', 'already', 'then', 'say...'], ['nah', 'i', "don't", 'think', 'he', 'goes', 'to', 'usf,', 'he', 'lives', 'around', 'here', 'though'], ['even', 'my', 'brother', 'is', 'not', 'like', 'to', 'speak', 'with', 'me.', 'they', 'treat', 'me', 'like', 'aids', 'patent.'], ['as', 'per', 'your', 'request', "'melle", 'melle', '(oru', 'minnaminunginte', 'nurungu', "vettam)'", 'has', 'been', 'set', 'as', 'your', 'callertune', 'for', 'all', 'callers.', 'press', '*9', 'to', 'copy', 'your', 'friends', 'callertune'], ["i'm", 'gonna', 'be', 'home', 'soon', 'and', 'i', "don't", 'want', 'to', 'talk', 'about', 'this', 'stuff', 'anymore', 'tonight,', 'k?', "i've", 'cried

In [64]:
# Viewing some of the spam data...
print(data['spam'][2])
print(data['spam'][4])

['winner!!', 'as', 'a', 'valued', 'network', 'customer', 'you', 'have', 'been', 'selected', 'to', 'receivea', '£900', 'prize', 'reward!', 'to', 'claim', 'call', '09061701461.', 'claim', 'code', 'kl341.', 'valid', '12', 'hours', 'only.']
['six', 'chances', 'to', 'win', 'cash!', 'from', '100', 'to', '20,000', 'pounds', 'txt>', 'csh11', 'and', 'send', 'to', '87575.', 'cost', '150p/day,', '6days,', '16+', 'tsandcs', 'apply', 'reply', 'hl', '4', 'info']


### **1.1 Prior Probability**

Let's look at our equation.

$$P(C|m) = \frac{P(m|C) P(C) }{P(m)}$$


Let's start with the easiest term - $P(C)$. This simply represents the probability that a message - out of *all* the e-mails an inbox receives - is spam or legitimate. This is the *prior probability*. This is important because it sets our baseline for what we're expecting, regardless of the actual message content. 

For example, if someone has an e-mail account that they only use to sign up for promotions and nothing important, then $P(Spam)$ would be high. Therefore, even if the message looks legitimate, since we come in with the belief that the inbox will be majority spam in the first place, we'll estimate that incoming message to be spam. 

Conversely, if someone is very careful about giving out their personal e-mail and only gives it to trusted friends, and thus very rarely gets spam in the first place, then $P(Spam)$ would be low. In this case, a spam-looking e-mail could arrive, but since we do not expect that many e-mails arriving in this inbox to be spam in the first place, our estimate that this particular e-mail is spam would be lower.

We could pull a number out of thin air, but if we had a dataset that represented somebody's inbox where someone has already labeled messages as spam or legitimate - we can calculate the fraction of e-mails that are spam and legitimate. This will serve as our estimate for $P(C)$. You'll do this now.

#### **Exercise 1.1: Estimating the Prior.**

Write code that estimates the prior for a given `class_value` ("spam" or "legit") given a data dictionary `dataset`. You can estimate $P(C)$ by calculating the ratio of messages in the dataset that are of class $c$ over the total number of messages in the dataset.

In [65]:
def get_prior(class_value, dataset):
    prior_prob = len(dataset[class_value]) / (len(dataset['legit']) + len(dataset['spam']))
    return prior_prob

In [66]:
grader.check("q_prior")

q_prior results: All test cases passed!

#### **Exercise 1.2: Estimating the Likelihood.**

One term down, two more to go.

$$P(C|m) = \frac{P(m|C) P(C) }{P(m)}$$

Now, let's see how we could calculate $P(m|C)$. This is a bit trickier and we'll have to make some assumptions.


##### **Bag of Words**
$P(m | C)$ asks: "Given that I know the class of a message (e.g. spam or legit), what's the probability of this message occurring?" 

"The probability of this message occurring" is underspecified and could have multiple interpretations. Let's try to specify this further, making some assumptions along the way.

The first simplifying assumption that we'll make is by using a version of a [*bag of words*](https://en.wikipedia.org/wiki/Bag-of-words_model) model. In a bag of words model, we disregard the order of the words and only care about their *frequency*. To make it even more simple for this example, we won't even care about frequency of a word - only if the word is present in the message or not. We'll say that a message $m$ is just an unordered list ("bag") of $n$ unique words, $(w_1, w_2, ..., w_n)$. Therefore:

$$ P(m | C) = P(w_1, w_2, ..., w_n | C)$$


This means that we'll interpret $P(m | C)$ to mean "the probability that a message consists only of these words." or "the probability of word $w_1$ and $w_2$ ... and $w_n$".  Again, this removes any notion of order or how often a certain word appears, which is not true to reality - but ends up still working quite well. 

Under this assumption, the following messages $m$ would all have the *same* $P(m | C)$, as they all contain (and only consist of) the words "free", "food", and "today".

- "free food today"
- "food today free"
- "today free food"
- "free free food today"
- "today food free free free"




##### **Conditional Independence**

Even though we've more clearly defined what $P(m | C)$ means, we are still unsure of how we would actually calculate it. For example, what if we see a combination of words we've never seen before? We are going to make one final assumption to make the problem tractable. We will assume that given the class, the probability of each word is *independent* of every other word. This means that:

$$P(m | C) = P(w_1, w_2, ..., w_n | C) = P(w_1 | C)P(w_2 | C)... P(w_n | C)$$

or, more succinctly,

$$P(m | C) = P(w_1, w_2, ..., w_n | C) = \prod_{i=1}^{n} P(w_i | C)$$



This is, of course, false - as the presence of one word sometimes implies the presence of another word. For example, in spam class, you could imagine that the word "prize" *always* jointly occurs with the word "free" so assuming $P(free, prize) = P(free)P(prize)$ is invalid.


However, this vastly simplifies the calculations that we need to do. Now for each word $w_i \in m$, we can estimate $P(w_i | C)$ by simply calculating the ratio of how many messages in $C$ contain $w_i$ over the total number of messages in $C$.

For example, consider if we have 4 spam messages:

- "Free prize if you respond now!"
- "Free samples this weekend only!"
- "Did you know you could get a prize for free?"
- "I have a secret for you... message me..."

$P(free | Spam) = 3/4$, as "free" is present in 3 out of the 4 messages.

$P(prize | Spam) = 2/4$, as "prize" is present in 2 out of the 4 messages.

Therefore, under the independence assumption:

$$P(free\ prize | Spam) = P(free, prize | Spam) = P(free|Spam)P(prize|Spam) = \frac{3}{4}*\frac{2}{4} = 0.375$$

Notice how this understates the true probability of $P(free, prize | Spam)$, which is $\frac{1}{2}$ (as "free" and "prize" occur together in 2/4 of the messages, because "prize" always comes with "free"). This is a trade-off that we will live with.

##### **Smoothing**

There's only one more problem that we have to address - what if we encounter a word that's not present in the dataset? In our current formulation, that would mean that $P(w_i | C) = 0$ for both classes which breaks our classifier, as it will zero out everything else in $P(w_1 | C)P(w_2 | C)... P(w_n | C)$. 

To get around this problem, if a word is not present in the class, we default to giving the unseen word a low but non-zero probability $\alpha$, such as $\alpha = 1/1000$. This process is called *smoothing*.

#### **Exercise 1.3: Implementing the Conditional Probability.**

Implement `conditional_prob` that estimates, for a single word $w$, the probability that it is in class $C$ - i.e., estimates $P(w | C)$. You can do this by simply calculating the ratio of how many messages in $C$ contain $w$ over the total number of messages in $C$. If $w$ is not present in $C$, the probability is instead the `smoothing_constant`.

In [67]:
def conditional_prob(word, class_value, dataset, smoothing_constant = 1/1000):
    containW = 0
    for data in dataset[class_value]:
        if word in data:
            containW+=1
    if containW == 0:
        return smoothing_constant
    
    return containW / len(dataset[class_value])


In [68]:
# Let's compare the probability of two words
print("P(free | SPAM)")
print(conditional_prob("free", "spam", data))

print("P(free | LEGIT)")
print(conditional_prob("free", "legit", data))

P(free | SPAM)
0.20214190093708165
P(free | LEGIT)
0.009736896623161384


In [69]:
grader.check("q_conditional_prob")

q_conditional_prob results: All test cases passed!

<!-- BEGIN QUESTION -->

#### **Exercise 1.4 (Written): (Not) Estimating the probability of the data.**

Two out of three terms down.

$$P(C|m) = \frac{P(m|C) P(C) }{P(m)}$$

Through conditional independence, we've transformed this into:

$$P(C|m) = \frac{P(C)\prod_{i=1}^{n} P(w_i | C) }{P(m)}$$

Now, the only thing we have to worry about is that pesky $P(m)$. What's the probability that this message occurred in our data *regardless* of its class? If we want to calculate $P(C | m)$, we need to calculate this somehow.

The good news is that we're not *actually* interested in calculating $P(C | m)$ !

Remember that our problem statement is that we are building a *classifier*. If we just want to classify an instance, we only want to know which is bigger - $P(Spam | m)$ or $P(Legit | m)$. If we're just interested in this, *we never have to calculate $P(m)$*.

In two or three sentences - explain how we can build a classifier that only outputs whether a message is more likely to be Spam or Legit without needing to calculate $P(m)$ (e.g., we only need $P(m|C)$ and $P(C)$).

$P(m)$ is the same for both $P(Spam | m)$ and $P(Legit | m)$ so they would effectively cancel out each other. Therefore, to compare these two probabilities we just need the values from their numerator. For $P(Spam | m)$, it is $P(m | Spam) P(Spam)$. For $P(Legit | m)$, it is $P(m | Legit) P(Legit)$.

<!-- END QUESTION -->

#### **Exercise 1.5: Building the Classifier.**

Using the functions you've written above (`get_prior` and `conditional_prob`), as well as your idea from the last question, fill out the function `classify` that takes in a message and determines if it is more likely spam or legit.

If it is more likely spam, it should return `spam`; if it is more likely legit, it should return `legit`. If they are (somehow) both the exact same probability, return `legit`.

**Note:** you'll notice that the probabilities you'll get will get very small very quickly (as you are multiplying many very small numbers). It shouldn't be an issue for small amount of text in messages so you don't need to do anything special. 

However, for problems with a lot of words (or other approaches to other problems), it's good to know that real life implementations usually work on the *logarithm* of the probabilities. If you're curious, try and think of reasons why the properties of the log function make this useful. (What happens when you take the log of a product? What happens to numbers between (0, 1) when you take the log?). 

In [70]:
def classify(message, data):
    classification = None
    prob_spam = get_prior("spam", dataset=data)
    prob_legit = get_prior("legit", dataset=data)
    for word in message:
        # spam
        prob_spam *= conditional_prob(word, "spam", dataset=data)
        # legit
        prob_legit *=  conditional_prob(word, "legit", dataset=data)
    if prob_spam > prob_legit:
        classification = 'spam'
    else: 
        classification = 'legit'
    return classification

In [71]:
# Let's classify some text!
classify("Hey there? How's it going i'm free today!", data)

'legit'

In [72]:
classify("Claim your free prize", data)

'legit'

In [73]:
classify("meeting today", data)

'legit'

In [74]:
grader.check("q_classify")

q_classify results: All test cases passed!

## **2. Bayes Nets**

In this part of the lab, you'll be using [pgmpy](https://pgmpy.org/index.html), which is a library for building and doing inference on graphical models (such as Bayes Nets). 

In [75]:
# First, let's do our imports
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling

Let's define a Bayesian Network where the variable C is influenced by both A and B (i.e. A -> C <- B). A and B are both parents of C.

Information is pulled from their [documentation pages](https://pgmpy.org/api/generated/models/pgmpy.models.DiscreteBayesianNetwork.html).

In [76]:

# A -> C and B -> C
abc_model = DiscreteBayesianNetwork([("A", "C"), ("B", "C")])


# Our conditional probability tables are called
# Conditional probability distributions (CPDs)

# Let's set up for the first node, A

# The first entry is P(A=0)
# The second is P(A=1)
# Notice that they add up to 1.


# +------+-----+
# | A(0) | 0.6 |
# +------+-----+
# | A(1) | 0.4 |
# +------+-----+
# E.g. P(A=0) = 0.6
cpd_a = TabularCPD("A", 2, [[0.6], [0.4]])

# Do the same for P(B)

# +------+-----+
# | B(0) | 0.7 |
# +------+-----+
# | B(1) | 0.3 |
# +------+-----+
# E.g. P(B=1) = 0.3
cpd_b = TabularCPD("B", 2, [[0.7], [0.3]])


# P(C | A,B) is a little more involved
# But see the table here to reference how we're 
# entering the probabilities

# E.g. P(C = 0 | A = 0, B=0) = 0.9
# E.g. P(C = 1 | A = 0, B=0) = 0.1

# E.g. P(C = 0 | A = 0, B=1) = 0.6
# Etc.

# +------+------+------+------+------+
# | A    | A(0) | A(0) | A(1) | A(1) |
# +------+------+------+------+------+
# | B    | B(0) | B(1) | B(0) | B(1) |
# +------+------+------+------+------+
# | C(0) | 0.9  | 0.6  | 0.7  | 0.1  |
# +------+------+------+------+------+
# | C(1) | 0.1  | 0.4  | 0.3  | 0.9  |
# +------+------+------+------+------+



cpd_c = TabularCPD(
    variable="C",
    variable_card=2,
    values=[
        [0.9, 0.6, 0.7, 0.1], #P(C=0 | A,B)
        [0.1, 0.4, 0.3, 0.9]  #P(C=1 | A,B)
        ],
    evidence=["A", "B"],
    evidence_card=[2, 2],
)

# Add our CPDs to the network
abc_model.add_cpds(cpd_a, cpd_b, cpd_c)

assert abc_model.check_model()

Now that we have our network, pgmpy has ways to do *exact inference*. This involves applications of Bayes' rule beyond the scope of the course, but you should know that it can be done. This is done via [VariableElimination](https://pgmpy.org/api/generated/inference/pgmpy.inference.VariableElimination.html).

In [77]:
# If we get P(A), we should expect the exact same as we specified in `abc_model`
from pgmpy.inference import VariableElimination

exact = VariableElimination(abc_model).query(["A"])
print(exact)

+------+----------+
| A    |   phi(A) |
+======+==========+
| A(0) |   0.6000 |
+------+----------+
| A(1) |   0.4000 |
+------+----------+


In [78]:
# However, let's say we just want to know P(C), regardless of A and B
# That requires some Bayesian Thinking, but luckily we can just ask pgmpy to do it for us!
exact = VariableElimination(abc_model).query(["C"])
print(exact)

+------+----------+
| C    |   phi(C) |
+======+==========+
| C(0) |   0.6940 |
+------+----------+
| C(1) |   0.3060 |
+------+----------+


Now, we can also *approximate* this via sampling methods. Let's sample from this Bayes Net. We use [BayesianModelSampling](https://pgmpy.org/api/generated/inference/pgmpy.sampling.BayesianModelSampling.html).

In [79]:
from pgmpy.sampling import BayesianModelSampling

sampler = BayesianModelSampling(abc_model)

# Let's generate 10 samples
num_samples = 10
samples = sampler.forward_sample(size=num_samples, seed=42)

samples

  0%|          | 0/3 [00:00<?, ?it/s]

,A,C,B
0,0,0,0
1,1,1,1
2,1,1,1
3,0,0,0
4,0,0,0
5,0,0,0
6,0,0,0
7,1,0,0
8,1,0,0
9,1,0,0


These are 10 samples that were generated from the Bayes Net we specified.

`samples` is a *dataframe*: a table-like structure that makes calculations easy. You've already seen dataframes a small bit in the prior probability lab (no pun intended).

Now, we can count the number of samples to get an estimate of the probability.

In [80]:
print("Num Samples: ", num_samples)
print("Estimated P(C = 0)")
print(sum(samples["C"]==0)/num_samples)

print("Estimated P(C = 1)")
print(sum(samples["C"]==1)/num_samples)

Num Samples:  10
Estimated P(C = 0)
0.8
Estimated P(C = 1)
0.2


Not a very good estimate. However, notice that as we take more and more samples, we approach the exact probability that we calculated earlier.

In [81]:
from pgmpy.sampling import BayesianModelSampling

sampler = BayesianModelSampling(abc_model)

for num_samples in [10,100,1000,10000, 100000]:
    samples = sampler.forward_sample(size=num_samples, seed=42)

    print("Num Samples: ", num_samples)
    print("Estimated P(C = 0)")
    print(sum(samples["C"]==0)/num_samples)

    print("Estimated P(C = 1)")
    print(sum(samples["C"]==1)/num_samples)

  0%|          | 0/3 [00:00<?, ?it/s]

Num Samples:  10
Estimated P(C = 0)
0.8
Estimated P(C = 1)
0.2


  0%|          | 0/3 [00:00<?, ?it/s]

Num Samples:  100
Estimated P(C = 0)
0.71
Estimated P(C = 1)
0.29


  0%|          | 0/3 [00:00<?, ?it/s]

Num Samples:  1000
Estimated P(C = 0)
0.69
Estimated P(C = 1)
0.31


  0%|          | 0/3 [00:00<?, ?it/s]

Num Samples:  10000
Estimated P(C = 0)
0.6945
Estimated P(C = 1)
0.3055


  0%|          | 0/3 [00:00<?, ?it/s]

Num Samples:  100000
Estimated P(C = 0)
0.69463
Estimated P(C = 1)
0.30537


### The Sprinkler Network

Consider the classic **sprinkler network** below - a common example problem used to explain Bayes Nets. `Cloudy` influences both `Sprinkler` and `Rain`, and `Sprinkler` and `Rain` both influence `WetGrass`. 

Note: One thing you should know is that sometimes when some people specify conditional probability tables, they specify only the probability that the value is true and use the complement rule to get the other value. For example, $P(C)=0.5$ here means that the probability of C being true is 0.5. We can then use the complement rule to determine that the probability of C being false is also 0.5.

Similarly, for $P(S \mid C)$. This table says that $P(S = T \mid C = T) = 0.1$, so thus $P(S = F \mid C = T)$ must be $0.9$ via the complement rule.

![Sprinkler network](sprinkler.jpg)

The next cell builds this network. The CPDs are provided, and you'll use the network in the exercise at the end of this section. We'll be mapping T to 1 and F to 0.

In [82]:
sprinkler_model = DiscreteBayesianNetwork([
    ("Cloudy", "Sprinkler"),
    ("Cloudy", "Rain"),
    ("Sprinkler", "WetGrass"),
    ("Rain", "WetGrass"),
])

# +-----------+-----+
# | Cloudy(0) | 0.5 |
# +-----------+-----+
# | Cloudy(1) | 0.5 |
# +-----------+-----+
# P(Cloudy)
cpd_cloudy = TabularCPD("Cloudy", 2, [[0.5], [0.5]])

# +--------------+-----------+-----------+
# | Cloudy       | Cloudy(0) | Cloudy(1) |
# +--------------+-----------+-----------+
# | Sprinkler(0) | 0.5       | 0.9       |
# +--------------+-----------+-----------+
# | Sprinkler(1) | 0.5       | 0.1       |
# +--------------+-----------+-----------+
# P(Sprinkler | Cloudy):   col 0 -> Cloudy=0, col 1 -> Cloudy=1
cpd_sprinkler = TabularCPD(
    "Sprinkler", 2,
    [[0.5, 0.9],   # P(Sprinkler=0 | Cloudy)
     [0.5, 0.1]],  # P(Sprinkler=1 | Cloudy)
    evidence=["Cloudy"], evidence_card=[2],
)

# +---------+-----------+-----------+
# | Cloudy  | Cloudy(0) | Cloudy(1) |
# +---------+-----------+-----------+
# | Rain(0) | 0.8       | 0.2       |
# +---------+-----------+-----------+
# | Rain(1) | 0.2       | 0.8       |
# +---------+-----------+-----------+
# P(Rain | Cloudy)
cpd_rain = TabularCPD(
    "Rain", 2,
    [[0.8, 0.2],
     [0.2, 0.8]],
    evidence=["Cloudy"], evidence_card=[2],
)

# +-------------+--------------+--------------+--------------+--------------+
# | Sprinkler   | Sprinkler(0) | Sprinkler(0) | Sprinkler(1) | Sprinkler(1) |
# +-------------+--------------+--------------+--------------+--------------+
# | Rain        | Rain(0)      | Rain(1)      | Rain(0)      | Rain(1)      |
# +-------------+--------------+--------------+--------------+--------------+
# | WetGrass(0) | 1.0          | 0.1          | 0.1          | 0.01         |
# +-------------+--------------+--------------+--------------+--------------+
# | WetGrass(1) | 0.0          | 0.9          | 0.9          | 0.99         |
# +-------------+--------------+--------------+--------------+--------------+
# P(WetGrass | Sprinkler, Rain)

cpd_wetgrass = TabularCPD(
    "WetGrass", 2,
    [[1.0, 0.1, 0.1, 0.01],
     [0.0, 0.9, 0.9, 0.99]],
    evidence=["Sprinkler", "Rain"], evidence_card=[2, 2],
)

sprinkler_model.add_cpds(cpd_cloudy, cpd_sprinkler, cpd_rain, cpd_wetgrass)

assert sprinkler_model.check_model()

Let's try to do a more complex query. Given that we know it is *not* rainy, what's the probability that the grass is wet? Again, we can use [`VariableElimination` ](https://pgmpy.org/api/generated/inference/pgmpy.inference.VariableElimination.html) to calculate the exact probability.

In [83]:
cond_exact = VariableElimination(sprinkler_model).query(
    ["WetGrass"], evidence={"Rain": 0}
)

print(cond_exact)

+-------------+-----------------+
| WetGrass    |   phi(WetGrass) |
+=============+=================+
| WetGrass(0) |          0.6220 |
+-------------+-----------------+
| WetGrass(1) |          0.3780 |
+-------------+-----------------+


### Rejection Sampling

Rejection sampling draws from the joint distribution, discards the samples that are inconsistent with the evidence, and estimates the conditional probability from the remaining samples. In this part, you will use `BayesianModelSampling` to implement it.

In [84]:
from pgmpy.sampling import BayesianModelSampling

sampler = BayesianModelSampling(sprinkler_model)

num_samples = 20
samples = sampler.forward_sample(size=num_samples, seed=42)

  0%|          | 0/4 [00:00<?, ?it/s]

One last thing you should know in order to do this is that if you want to *subset* a dataframe where variables have a certain value, you can do so with the following syntax (see [this page](https://pandas.pydata.org/docs/user_guide/indexing.html) for more).

In [85]:
# Get all rows where Rain = 0
not_raining_samples = samples[samples["Rain"] == 0]
not_raining_samples

,Cloudy,Sprinkler,Rain,WetGrass
0,0,1,0,1
1,1,0,0,0
3,1,1,0,1
5,0,0,0,0
13,0,1,0,1
15,0,1,0,1
16,0,1,0,1
18,0,0,0,0
19,0,1,0,1


You can also subset multiple variables at once.

In [86]:
cloudy_and_wet = samples[ (samples["Cloudy"] == 1) & (samples["WetGrass"] == 1)]
cloudy_and_wet 

,Cloudy,Sprinkler,Rain,WetGrass
2,1,0,1,1
3,1,1,0,1
7,1,1,1,1
8,1,0,1,1
9,1,0,1,1
11,1,0,1,1
12,1,0,1,1
17,1,0,1,1


In [87]:
not_raining_samples = samples[(samples["Rain"] == 0) & (samples['Cloudy'] == 1)]
not_raining_samples

,Cloudy,Sprinkler,Rain,WetGrass
1,1,0,0,0
3,1,1,0,1


<!-- BEGIN QUESTION -->

#### **Exercise 2.1 (Written): Rejection Sampling.**

For each query, (a) compute the exact probability with `VariableElimination`, (b) estimate it with rejection sampling using 10, 100, 1,000, 10,000, and 100,000 samples, and (c) record both results.

1. $P(WetGrass = 1 \mid Rain = 0)$
2. $P(WetGrass = 1 \mid Sprinkler = 1)$
3. $P(WetGrass = 1 \mid Cloudy = 1, Rain = 0)$

Then answer the following questions:

- Which queries does rejection sampling converge on faster? Which converge more slowly?
- For the queries that converge more slowly, explain why.
- Why does the rarity of the evidence matter for rejection sampling?

In [88]:
cond_exact = VariableElimination(sprinkler_model).query(
    ["WetGrass"], evidence={"Rain": 0}
)

print(cond_exact)

+-------------+-----------------+
| WetGrass    |   phi(WetGrass) |
+=============+=================+
| WetGrass(0) |          0.6220 |
+-------------+-----------------+
| WetGrass(1) |          0.3780 |
+-------------+-----------------+


In [89]:
cond_exact = VariableElimination(sprinkler_model).query(
    ["WetGrass"], evidence={"Sprinkler": 1}
)

print(cond_exact)

+-------------+-----------------+
| WetGrass    |   phi(WetGrass) |
+=============+=================+
| WetGrass(0) |          0.0730 |
+-------------+-----------------+
| WetGrass(1) |          0.9270 |
+-------------+-----------------+


In [90]:
cond_exact = VariableElimination(sprinkler_model).query(
    ["WetGrass"], evidence={"Rain": 0, "Cloudy": 1}
)

print(cond_exact)

+-------------+-----------------+
| WetGrass    |   phi(WetGrass) |
+=============+=================+
| WetGrass(0) |          0.9100 |
+-------------+-----------------+
| WetGrass(1) |          0.0900 |
+-------------+-----------------+


In [91]:
from pgmpy.sampling import BayesianModelSampling

sampler = BayesianModelSampling(sprinkler_model)

num_samples = 10
samples = sampler.forward_sample(size=num_samples, seed=42)

# wetgrass = 1 | rain = 0
rain = samples[(samples["Rain"] == 0)]
rain_and_wet = (rain["WetGrass"] == 1).mean()
print(rain_and_wet)

# wetgrass = 1 | sprinker = 1
sprinker = samples[(samples["Sprinkler"] == 1)]
sprinkler_and_wet = (sprinker["WetGrass"] == 1).mean()
print(sprinkler_and_wet)

# wetgrass = 1 | cloudy = 1, rain = 0
cloudy_and_not_rain = samples[(samples["Cloudy"] == 1) & (samples["Rain"] == 0)]
cloudy_not_rain_and_wet = (cloudy_and_not_rain["WetGrass"] == 1).mean()
print(cloudy_not_rain_and_wet)

  0%|          | 0/4 [00:00<?, ?it/s]

0.2
0.5
0.0


In [92]:
from pgmpy.sampling import BayesianModelSampling

sampler = BayesianModelSampling(sprinkler_model)

num_samples = 100
samples = sampler.forward_sample(size=num_samples, seed=42)

# wetgrass = 1 | rain = 0
rain = samples[(samples["Rain"] == 0)]
rain_and_wet = (rain["WetGrass"] == 1).mean()
print(rain_and_wet)

# wetgrass = 1 | sprinker = 1
sprinker = samples[(samples["Sprinkler"] == 1)]
sprinkler_and_wet = (sprinker["WetGrass"] == 1).mean()
print(sprinkler_and_wet)

# wetgrass = 1 | cloudy = 1, rain = 0
cloudy_and_not_rain = samples[(samples["Cloudy"] == 1) & (samples["Rain"] == 0)]
cloudy_not_rain_and_wet = (cloudy_and_not_rain["WetGrass"] == 1).mean()
print(cloudy_not_rain_and_wet)

  0%|          | 0/4 [00:00<?, ?it/s]

0.4117647058823529
0.9285714285714286
0.18181818181818182


In [93]:
from pgmpy.sampling import BayesianModelSampling

sampler = BayesianModelSampling(sprinkler_model)

num_samples = 1000
samples = sampler.forward_sample(size=num_samples, seed=42)

# wetgrass = 1 | rain = 0
rain = samples[(samples["Rain"] == 0)]
rain_and_wet = (rain["WetGrass"] == 1).mean()
print(rain_and_wet)

# wetgrass = 1 | sprinker = 1
sprinker = samples[(samples["Sprinkler"] == 1)]
sprinkler_and_wet = (sprinker["WetGrass"] == 1).mean()
print(sprinkler_and_wet)

# wetgrass = 1 | cloudy = 1, rain = 0
cloudy_and_not_rain = samples[(samples["Cloudy"] == 1) & (samples["Rain"] == 0)]
cloudy_not_rain_and_wet = (cloudy_and_not_rain["WetGrass"] == 1).mean()
print(cloudy_not_rain_and_wet)

  0%|          | 0/4 [00:00<?, ?it/s]

0.39553752535496955
0.9120521172638436
0.10416666666666667


In [94]:
from pgmpy.sampling import BayesianModelSampling

sampler = BayesianModelSampling(sprinkler_model)

num_samples = 10000
samples = sampler.forward_sample(size=num_samples, seed=42)

# wetgrass = 1 | rain = 0
rain = samples[(samples["Rain"] == 0)]
rain_and_wet = (rain["WetGrass"] == 1).mean()
print(rain_and_wet)

# wetgrass = 1 | sprinker = 1
sprinker = samples[(samples["Sprinkler"] == 1)]
sprinkler_and_wet = (sprinker["WetGrass"] == 1).mean()
print(sprinkler_and_wet)

# wetgrass = 1 | cloudy = 1, rain = 0
cloudy_and_not_rain = samples[(samples["Cloudy"] == 1) & (samples["Rain"] == 0)]
cloudy_not_rain_and_wet = (cloudy_and_not_rain["WetGrass"] == 1).mean()
print(cloudy_not_rain_and_wet)

  0%|          | 0/4 [00:00<?, ?it/s]

0.3849198495943004
0.9270867700428901
0.09753483386923902


In [95]:
from pgmpy.sampling import BayesianModelSampling

sampler = BayesianModelSampling(sprinkler_model)

num_samples = 100000
samples = sampler.forward_sample(size=num_samples, seed=42)

# wetgrass = 1 | rain = 0
rain = samples[(samples["Rain"] == 0)]
rain_and_wet = (rain["WetGrass"] == 1).mean()
print(rain_and_wet)

# wetgrass = 1 | sprinker = 1
sprinker = samples[(samples["Sprinkler"] == 1)]
sprinkler_and_wet = (sprinker["WetGrass"] == 1).mean()
print(sprinkler_and_wet)

# wetgrass = 1 | cloudy = 1, rain = 0
cloudy_and_not_rain = samples[(samples["Cloudy"] == 1) & (samples["Rain"] == 0)]
cloudy_not_rain_and_wet = (cloudy_and_not_rain["WetGrass"] == 1).mean()
print(cloudy_not_rain_and_wet)

  0%|          | 0/4 [00:00<?, ?it/s]

0.37942205779422056
0.9270629579285881
0.0888046298144083


(a).
1. $P(WetGrass = 1 \mid Rain = 0) = 0.622$
2. $P(WetGrass = 1 \mid Sprinkler = 1) = 0.927$
3. $P(WetGrass = 1 \mid Cloudy = 1, Rain = 0) = 0.090$

(b).
10:
1. $P(WetGrass = 1 \mid Rain = 0) = 0.2$
2. $P(WetGrass = 1 \mid Sprinkler = 1) = 0.5$
3. $P(WetGrass = 1 \mid Cloudy = 1, Rain = 0) = 0.0$

100:
1. $P(WetGrass = 1 \mid Rain = 0) = 0.412$
2. $P(WetGrass = 1 \mid Sprinkler = 1) = 0.929$
3. $P(WetGrass = 1 \mid Cloudy = 1, Rain = 0) = 0.182$

1,000:
1. $P(WetGrass = 1 \mid Rain = 0) = 0.396$
2. $P(WetGrass = 1 \mid Sprinkler = 1) = 0.912$
3. $P(WetGrass = 1 \mid Cloudy = 1, Rain = 0) = 0.104$

10,000:
1. $P(WetGrass = 1 \mid Rain = 0) = 0.385$
2. $P(WetGrass = 1 \mid Sprinkler = 1) = 0.927$
3. $P(WetGrass = 1 \mid Cloudy = 1, Rain = 0) = 0.0975$

100,000: 
1. $P(WetGrass = 1 \mid Rain = 0) = 0.379$
2. $P(WetGrass = 1 \mid Sprinkler = 1) = 0.927$
3. $P(WetGrass = 1 \mid Cloudy = 1, Rain = 0) = 0.0888$

- Wetgrass given sprinker converges the fastest out of the 3 since it only took 10,000 to produce the same result. Followed by Rain and then cloudy. 
- I think they differ because of the probability of them occuring. Wetgrass given sprinker is the most likely with 0.93 chance since if the sprinkler is on it is very likely the grass is wet. Rain varies slightly because the chance of it happening is rarer while cloudy and not raining is the least likely to converge since it is the rarest and therefore has little evidence.
- Rejection sampling discards every row that fails to match the evidence. With 100 generated samples, query 3 therefore uses only about 10 rows on average to estimate the probability. With fewer accepted rows, each wet or dry outcome has a larger effect on the estimate. If zero rows match, the estimate is undefined.

<!-- END QUESTION -->

## **Collaboration Statement**

You are required to complete the following collaboration statement.

<!-- BEGIN QUESTION -->

- If you completed the lab in a group - list all your group members here, including yourself. **If you worked individually, you must state *"I worked by myself."***
    - I worked by myself.
- You must list the names of any others you got help from when completing this assignment (including TAs), and in what way you worked with them (discussed ideas, debugged math). **If you did not collaborate at all, you must state *"I did not collaborate with anyone else."***
    - I did not collaborate with anyone else.
- If you found any resources online helpful, you must link them and describe how you found them helpful (or not helpful). **If you did not use any resources online, you must state *"I did not use any resources online."***
    - I did not use any resources online.
- If you used any AI assistive technologies, you must describe how you used them and how you found them helpful (or not helpful). **If you did not use any AI assistive technologies, you must state *"I did not use any AI assistive technologies."***
    - I did not use any AI assistive technologies.
- If you have any questions or comments about the material, or there was a topic or question you are confused about -- please make a note of it here! **If you do not have any questions or comments, you must state *"I do not have any questions or comments about the assignment."***
    - I do not have any questions or comments about the assignment."
- Estimate how long this assignment took you in hours. **You must give an estimate**.
    - This assignment took me about 4 hours to complete.


<!-- END QUESTION -->



## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Run every cell from top to bottom and SAVE before exporting. Then submit
the generated .zip to the <HWN> autograder assignment on Gradescope.


In [96]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)

Running your submission against local test cases...


Your submission received the following results when run against available test cases:

    q_prior results: All test cases passed!

    q_conditional_prob results: All test cases passed!

    q_classify results: All test cases passed!


/Users/arthur/Downloads/Git/cs31_env/.venv/lib/python3.13/site-packages/otter/check/utils.py:115: RuntimeWarning: [IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.

  warnings.warn(results.stderr.decode("utf-8"), RuntimeWarning)
